In [100]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score)
from sklearn.metrics import confusion_matrix

In [75]:
df=pd.read_parquet('../data/featured/olist_delivery_features.parquet')

In [76]:
df.head()

,late_delivery,purchase_hour,purchase_month,expected_delivery_days,approval_hours,Monday,Saturday,Sunday,Thursday,Tuesday,Wednesday
0,0,10,10,15,0.178333,1,0,0,0,0,0
1,0,20,7,19,30.713889,0,0,0,0,1,0
2,0,8,8,26,0.276111,0,0,0,0,0,1
3,0,19,11,26,0.298056,0,1,0,0,0,0
4,0,21,2,12,1.030556,0,0,0,0,1,0


### Training a Baseline Model
- Logistic Regression
    - Simple
    - Fast
    - Probabilities
    - Good baseline
Establish a benchmark

In [77]:
X=df.drop('late_delivery',axis=1)
Y=df.late_delivery

In [78]:
X.head()

,purchase_hour,purchase_month,expected_delivery_days,approval_hours,Monday,Saturday,Sunday,Thursday,Tuesday,Wednesday
0,10,10,15,0.178333,1,0,0,0,0,0
1,20,7,19,30.713889,0,0,0,0,1,0
2,8,8,26,0.276111,0,0,0,0,0,1
3,19,11,26,0.298056,0,1,0,0,0,0
4,21,2,12,1.030556,0,0,0,0,1,0


In [79]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 96461 entries, 0 to 96460
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   purchase_hour           96461 non-null  int32  
 1   purchase_month          96461 non-null  int32  
 2   expected_delivery_days  96461 non-null  int64  
 3   approval_hours          96461 non-null  float64
 4   Monday                  96461 non-null  int64  
 5   Saturday                96461 non-null  int64  
 6   Sunday                  96461 non-null  int64  
 7   Thursday                96461 non-null  int64  
 8   Tuesday                 96461 non-null  int64  
 9   Wednesday               96461 non-null  int64  
dtypes: float64(1), int32(2), int64(7)
memory usage: 6.6 MB


In [80]:
Y.head()

0    0
1    0
2    0
3    0
4    0
Name: late_delivery, dtype: int64

In [81]:
x_train,x_test,y_train,y_test= train_test_split(X,Y,test_size=.20,random_state=20,stratify=Y)

In [85]:
num_cols=['purchase_hour','purchase_month','expected_delivery_days','approval_hours']

In [86]:
scaler=StandardScaler()
x_train[num_cols]=scaler.fit_transform(x_train[num_cols])
x_test[num_cols]=scaler.fit_transform(x_test[num_cols])

In [87]:
x_train.head()

,purchase_hour,purchase_month,expected_delivery_days,approval_hours,Monday,Saturday,Sunday,Thursday,Tuesday,Wednesday
42860,-0.142572,-0.629950,3.269043,2.522973,0,1,0,0,0,0
32691,0.419117,0.299397,-0.611879,-0.483274,0,0,1,0,0,0
50506,0.231887,1.228745,-0.269444,1.861106,1,0,0,0,0,0
15676,0.793576,0.918963,-0.840168,-0.487962,1,0,0,0,0,0
62062,-1.453179,0.609180,1.100292,-0.489791,0,0,0,0,0,1


In [90]:
lr=LogisticRegression(random_state=20,max_iter=1000)
lr.fit(x_train,y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,20
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [105]:
y_pred=lr.predict(x_test) # predict() , threshold >=0.5 -> 1
y_prob=lr.predict_proba(x_test)[:,1]

In [106]:
y_prob

array([0.11202341, 0.04726428, 0.07435699, ..., 0.08554969, 0.05382263,
       0.0573057 ], shape=(19293,))

In [107]:
print("Accuracy :", accuracy_score(y_test,y_pred))
print("Precision:", precision_score(y_test,y_pred))
print("Recall   :", recall_score(y_test,y_pred))
print("F1 Score :", f1_score(y_test,y_pred))
print("ROC AUC  :", roc_auc_score(y_test,y_prob))

Accuracy : 0.9189343285129321
Precision: 1.0
Recall   : 0.0006389776357827476
F1 Score : 0.001277139208173691
ROC AUC  : 0.5847508967601297


In [108]:
cm = confusion_matrix(y_test,y_pred)
print(cm)

[[17728     0]
 [ 1564     1]]


In [109]:
pd.Series(lr.predict_proba(x_test)[:,-1]).describe()

count    19293.000000
mean         0.081108
std          0.021339
min          0.002666
25%          0.066929
50%          0.079670
75%          0.093903
max          0.721018
dtype: float64

In [125]:
y2_prob=lr.predict_proba(x_test)[:,1]

In [ ]:
y2_pred=(y2_prob>=.2).astype(int)

In [129]:
print("Accuracy :", accuracy_score(y_test,y2_pred))
print("Precision:", precision_score(y_test,y2_pred))
print("Recall   :", recall_score(y_test,y2_pred))
print("F1 Score :", f1_score(y_test,y2_pred))
print("ROC AUC  :", roc_auc_score(y_test,y2_prob))

Accuracy : 0.9186751671590733
Precision: 0.25
Recall   : 0.0012779552715654952
F1 Score : 0.0025429116338207248
ROC AUC  : 0.5847508967601297


No significant improvements even after decreasing threshold.
- Weakness - Class Imbalance

In [137]:
lr_model2=LogisticRegression(random_state=20,max_iter=100,class_weight='balanced')
lr_model2.fit(x_train,y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,20
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [138]:
y3_pred=lr_model2.predict(x_test)

In [139]:
y3_prob=lr_model2.predict_proba(x_test)[:,1]

In [140]:
print("Accuracy :", accuracy_score(y_test,y3_pred))
print("Precision:", precision_score(y_test,y3_pred))
print("Recall   :", recall_score(y_test,y3_pred))
print("F1 Score :", f1_score(y_test,y3_pred))
print("ROC AUC  :", roc_auc_score(y_test,y3_prob))

Accuracy : 0.5439796817498574
Precision: 0.10060739922694643
Recall   : 0.582108626198083
F1 Score : 0.17156308851224106
ROC AUC  : 0.5838047571538967


In [142]:
cm_model2=confusion_matrix(y_test,y3_pred)
cm_model2

array([[9584, 8144],
       [ 654,  911]])